In [1]:
from hdx.utilities.easy_logging import setup_logging
from hdx.api.configuration import Configuration
from hdx.data.dataset import Dataset
import geopandas as gpd
import zipfile
import os

setup_logging()

In [5]:
class Hdx:
    def __init__(self):
        self.data = Dataset
        if not Configuration._configuration:
            Configuration.create(hdx_site='prod', user_agent='A_Quick_Example', hdx_read_only=True)
    def get_data(self, dataset_name, file_type):
            dataset = self.data.read_from_hdx(dataset_name)
            if dataset is None:
                print("Dataset not found.")
                return None

            resources = dataset.get_resources()

            target_resource = None
            for resource in resources:
                if resource['format'].lower() == file_type.lower():
                    target_resource = resource
                    break

            if target_resource:
                # Download the file to your local environment
                url, path = target_resource.download(folder="../data/hdx")
                print(f"Downloaded file to: {path}")

                # https://stackoverflow.com/questions/3451111/unzipping-files-in-python
                if zipfile.is_zipfile(path):
                    print("Extracting zip archive...")
                    with zipfile.ZipFile(path, 'r') as zip_ref:
                        extract_dir = os.path.dirname(path)
                        zip_ref.extractall(extract_dir)

                        extracted_files = zip_ref.namelist()
                        target_file = next((os.path.join(extract_dir, f) for f in extracted_files if f.endswith(('.geojson', '.shp', '.gpkg'))), None)

                        if target_file:
                            path = target_file
                        else:
                            print("Could not find a supported spatial file inside the zip.")
                            return None

                # Load the file into a GeoPandas GeoDataFrame without hardcoding the driver
                gdf = gpd.read_file(path)

                print("Successfully loaded into variable 'gdf'!")
                return gdf
            else:
                print("No suitable vector/boundary format found in resources.")
                return None

In [6]:
data_name = 'cod-ab-sdn'


In [8]:
hdx = Hdx()
test = hdx.get_data(dataset_name=data_name, file_type='geojson')

Downloaded file to: ..\data\hdx\sdn_admin_boundaries.geojson.zip.geojson
Extracting zip archive...
Successfully loaded into variable 'gdf'!


In [9]:
test

,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,...,area_sqkm,version,lang,lang1,lang2,lang3,adm2_ref_name,center_lat,center_lon,geometry
0,Abassiya,العباسية,None,None,SD07090,South Kordofan,جنوب كردفان,None,None,SD07,...,4231.346747,v03,en,ar,None,None,Abassiya,12.234464,31.271965,"POLYGON ((31.59964 12.61949, 31.59259 12.62135..."
1,Abu Hamad,أبو حمد,None,None,SD16008,River Nile,نهر النيل,None,None,SD16,...,31673.076420,v03,en,ar,None,None,Abu Hamad,20.153503,33.219586,"POLYGON ((33.55492 21.72171, 33.32621 21.88507..."
2,Abu Hujar,أبو حجار,None,None,SD14037,Sennar,سنار,None,None,SD14,...,3829.289207,v03,en,ar,None,None,Abu Hujar,12.669136,33.878764,"POLYGON ((34.00041 12.85744, 33.9974 12.86797,..."
3,Abu Jabrah,أبو جابرة,None,None,SD05140,East Darfur,شرق دارفور,None,None,SD05,...,6060.470729,v03,en,ar,None,None,Abu Jabrah,10.950397,26.868803,"POLYGON ((26.88396 10.53806, 27.02585 10.56789..."
4,Abu Jubayhah,أبو جبيهة,None,None,SD07088,South Kordofan,جنوب كردفان,None,None,SD07,...,20973.493722,v03,en,ar,None,None,Abu Jubayhah,11.002253,31.874989,"POLYGON ((32.12793 11.94509, 32.12408 11.94901..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,Wadi Salih,وادي صالح,None,None,SD06137,Central Darfur,وسط دارفور,None,None,SD06,...,3311.154967,v03,en,ar,None,None,Wadi Salih,12.416140,23.371006,"POLYGON ((23.25752 12.6562, 23.24825 12.67612,..."
185,Wasat Al Gedaref,وسط القضارف,None,None,SD12081,Gedaref,القضارف,None,None,SD12,...,9576.765906,v03,en,ar,None,None,Wasat Al Gedaref,14.151642,34.994637,"POLYGON ((35.6028 14.45613, 35.60077 14.53739,..."
186,Wasat Jabal Marrah,وسط جبل مرة,None,None,SD06139,Central Darfur,وسط دارفور,None,None,SD06,...,447.717853,v03,en,ar,None,None,Wasat Jabal Marrah,13.123652,24.313515,"POLYGON ((24.28869 13.19028, 24.27884 13.1946,..."
187,Yassin,يس,None,None,SD05165,East Darfur,شرق دارفور,None,None,SD05,...,5493.331388,v03,en,ar,None,None,Yassin,11.706977,25.586413,"POLYGON ((25.6472 12.21491, 25.60531 12.23928,..."


In [ ]:
dataset = Dataset.read_from_hdx('cod-ab-sdn')

if dataset:
    print("Dataset available.")

    # Iterate through the resources (files) attached to this dataset
    resources = dataset.get_resources()
    for resource in resources:
        print(f"Resource Name: {resource['name']} | Format: {resource['format']}")


        # If you want to download a specific resource file (e.g., a zipped shapefile or geodatabase):
        # file_path, folder = resource.download()
        # print(f"Downloaded to: {file_path}")
else:
    print("Dataset not found.")